In [0]:
from pyspark.sql import functions as F

base = "/Volumes/workspace/bronze/raw_landing"

for name in ["customers", "products"]:
    df = (spark.read
          .option("header", True)          # no inferSchema: keep everything as strings
          .csv(f"{base}/{name}/")
          .withColumn("_ingested_at", F.current_timestamp())
          .withColumn("_source_file", F.col("_metadata.file_path")))
    df.write.mode("overwrite").saveAsTable(f"workspace.bronze.{name}")
    print(name, df.count())

In [0]:
(spark.readStream.format("cloudFiles")
   .option("cloudFiles.format", "csv")
   .option("header", True)
   .option("cloudFiles.schemaLocation", "/Volumes/workspace/bronze/raw_landing/_schemas/orders")
   .load("/Volumes/workspace/bronze/raw_landing/orders/")
   .withColumn("_ingested_at", F.current_timestamp())
   .withColumn("_source_file", F.col("_metadata.file_path"))
   .writeStream
   .option("checkpointLocation", "/Volumes/workspace/bronze/raw_landing/_checkpoints/orders")
   .trigger(availableNow=True)
   .toTable("workspace.bronze.orders"))

In [0]:
%sql
SELECT * FROM workspace.bronze.orders LIMIT 10;
SELECT COUNT(*), COUNT(DISTINCT order_id) FROM workspace.bronze.orders;

In [0]:
%sql
SELECT COUNT(*), COUNT(DISTINCT order_id) FROM workspace.bronze.orders;